# Augmentation and Training Schedule Comparison

Compare augmentation profiles, optimizers, schedulers, and projection heads
while holding backbone (ResNet18), dataset (LeaData), label strategy
(external_item_id), and embedding dimension (256) fixed.

> **Warning:** Results are diagnostic only. LeaData is a public leather texture
> dataset, not a luxury authentication dataset.

## Run Experiments

From `auth-platform/`:

```bash
# Quick (2 epochs, 100 images)
make augmentation-schedule-comparison-leadata-quick

# Medium (original epoch counts, 300 images)
make augmentation-schedule-comparison-leadata-medium

# Full
make augmentation-schedule-comparison-leadata
```

In [ ]:
import json
from pathlib import Path

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

AS_DIR = Path("../experiments/outputs/augmentation-schedule-comparison")
summary_path = AS_DIR / "augmentation_schedule_comparison_summary.json"
print(f"Exists: {summary_path.exists()}")

In [ ]:
if not summary_path.exists():
    print("No results. Run: make augmentation-schedule-comparison-leadata-quick")
    data = None
    results = []
else:
    data = json.loads(summary_path.read_text())
    results = [r for r in data.get("results", []) if r.get("evaluation_metrics")]
    results.sort(key=lambda x: x["evaluation_metrics"].get("cluster_separation", 0), reverse=True)
    print(f"Results: {len(results)}  Failures: {len(data.get('failures', []))}")
    print()

    header = f"{'Aug':<8} {'Loss':<18} {'Ep':>3} {'ClustSep':>10} {'NN Agree':>10} {'R@1':>8} {'Time':>8}"
    print(header)
    print("-" * len(header))
    for r in results:
        em = r["evaluation_metrics"]
        print(f"{r.get('augmentation_profile', '?'):<8} "
              f"{r.get('loss', '?'):<18} "
              f"{r.get('epochs', ''):>3} "
              f"{em.get('cluster_separation', 0):>10.4f} "
              f"{em.get('nn_agreement_rate', 0):>9.1%} "
              f"{em.get('recall_at_1', 0):>8.4f} "
              f"{r.get('training_time_seconds', ''):>7}s")

In [ ]:
# Cluster separation by config
if results:
    labels_plot = [f"{r.get('augmentation_profile','?')} {r.get('loss','?')}\n{r.get('epochs','')}ep" for r in results]
    seps = [r["evaluation_metrics"].get("cluster_separation", 0) for r in results]

    fig, ax = plt.subplots(figsize=(10, max(4, len(results) * 0.6)))
    bars = ax.barh(labels_plot, seps, color="steelblue")
    ax.axvline(x=0.9418, color="red", linestyle="--", label="texture_baseline_v0.2.0")
    ax.set_xlabel("Cluster Separation")
    ax.set_title("Augmentation/Schedule Comparison")
    ax.legend()
    for bar, val in zip(bars, seps):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

## Notes / Conclusions

- Fixed: ResNet18 pretrained fine-tuned, 256d, external_item_id labels.
- Varied: augmentation (none/light/medium/strong), optimizer (adam/adamw),
  scheduler (none/cosine), projection head (256/512), epoch count (5/10).
- `texture_baseline_v0.2.0` (0.9418) is the reference line.

**These results are diagnostic only.** They do not prove authentication accuracy.